In [29]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import gc
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.compose import ColumnTransformer
from sklearn.base import TransformerMixin, BaseEstimator
from my_functions import *

from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import cross_validate
from sklearn.linear_model import LogisticRegression

sns.set_theme()
gc.enable()

# Main Data

In [2]:
main_data=pd.read_csv('initial_clean.csv', keep_default_na=False, na_values='')
main_data.drop(['DAYS_BIRTH', 'DAYS_EMPLOYED', 
                'DAYS_ID_PUBLISH', 'DAYS_LAST_PHONE_CHANGE', 
                'DAYS_REGISTRATION'], axis=1, inplace=True)
main_data.shape

(307511, 43)

### Dealing with Outliers

In [3]:
'''
contains all outlier marker columns
'''
outlier_frame=main_data.filter(regex='outlier', axis=1).copy()
outlier_frame['SK_ID_CURR']=main_data['SK_ID_CURR']
outlier_frame['TARGET']=main_data['TARGET']
outlier_frame.shape

(307511, 8)

In [4]:
# plot_all_encoded_vs_target(outlier_frame)

In [5]:
outlier_frame.columns

Index(['AMT_CREDIT_outlier', 'AMT_INCOME_TOTAL_outlier', 'AMT_ANNUITY_outlier',
       'AMT_GOODS_PRICE_outlier', 'DAYS_REGISTRATION_outlier',
       'DAYS_EMPLOYED_outlier', 'SK_ID_CURR', 'TARGET'],
      dtype='object')

In [6]:
clean1=main_data.loc[(main_data['AMT_CREDIT_outlier']==0)&
              (main_data['AMT_INCOME_TOTAL_outlier']==0)&
              (main_data['AMT_ANNUITY_outlier']==0)&
              (main_data['AMT_GOODS_PRICE_outlier']==0)&
              (main_data['DAYS_REGISTRATION_outlier']==0)&
              (main_data['DAYS_EMPLOYED_outlier']==0), :].copy()
clean1.drop(labels=outlier_frame.columns.difference(['SK_ID_CURR', 'TARGET']), axis=1, inplace=True)
clean1.loc[clean1['NAME_FAMILY_STATUS'].isnull(), 'NAME_FAMILY_STATUS']=clean1['NAME_FAMILY_STATUS'].mode().values[0]
clean1.shape

(306295, 37)

### Scaling

In [7]:
clean1.columns.sort_values()

Index(['AMT_ANNUITY', 'AMT_CREDIT', 'AMT_GOODS_PRICE', 'AMT_INCOME_TOTAL',
       'CODE_GENDER', 'EMERGENCYSTATE_MODE', 'EXT_SOURCE_1', 'EXT_SOURCE_2',
       'EXT_SOURCE_3', 'FLAG_DOCUMENT_3', 'FLAG_DOCUMENT_6', 'FLAG_EMP_PHONE',
       'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'FLAG_PHONE', 'FLAG_WORK_PHONE',
       'FONDKAPREMONT_MODE', 'HOUSETYPE_MODE', 'LIVE_CITY_NOT_WORK_CITY',
       'NAME_CONTRACT_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS',
       'NAME_HOUSING_TYPE', 'NAME_INCOME_TYPE', 'NAME_TYPE_SUITE',
       'OCCUPATION_TYPE', 'ORGANIZATION_TYPE', 'REG_CITY_NOT_LIVE_CITY',
       'REG_CITY_NOT_WORK_CITY', 'SK_ID_CURR', 'TARGET', 'WALLSMATERIAL_MODE',
       'YEARS_BIRTH', 'YEARS_EMPLOYED', 'YEARS_ID_PUBLISH',
       'YEARS_LAST_PHONE_CHANGE', 'YEARS_REGISTRATION'],
      dtype='object')

In [8]:
ct=ColumnTransformer(transformers=[('numerical', StandardScaler(), ['AMT_ANNUITY', 'AMT_CREDIT', 'AMT_INCOME_TOTAL', 'AMT_GOODS_PRICE',
                                                                    'YEARS_BIRTH', 'YEARS_ID_PUBLISH', 'YEARS_REGISTRATION', 
                                                                    'YEARS_EMPLOYED', 'YEARS_LAST_PHONE_CHANGE'])],
                     remainder='passthrough')
clean2=pd.DataFrame(ct.fit_transform(clean1))
clean2.columns=ct.get_feature_names_out()
clean2.rename(columns=lambda x: x[11:], inplace=True)
clean2.shape

(306295, 37)

### Imputing Null Values and Encoding Categorical Variables

In [9]:
clean2.columns

Index(['AMT_ANNUITY', 'AMT_CREDIT', 'AMT_INCOME_TOTAL', 'AMT_GOODS_PRICE',
       'YEARS_BIRTH', 'YEARS_ID_PUBLISH', 'YEARS_REGISTRATION',
       'YEARS_EMPLOYED', 'YEARS_LAST_PHONE_CHANGE', 'EXT_SOURCE_1',
       'EXT_SOURCE_2', 'EXT_SOURCE_3', 'TARGET', 'SK_ID_CURR', 'CODE_GENDER',
       'EMERGENCYSTATE_MODE', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY',
       'FONDKAPREMONT_MODE', 'HOUSETYPE_MODE', 'NAME_CONTRACT_TYPE',
       'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE',
       'NAME_INCOME_TYPE', 'NAME_TYPE_SUITE', 'OCCUPATION_TYPE',
       'ORGANIZATION_TYPE', 'WALLSMATERIAL_MODE', 'FLAG_DOCUMENT_3',
       'FLAG_DOCUMENT_6', 'FLAG_EMP_PHONE', 'FLAG_PHONE', 'FLAG_WORK_PHONE',
       'LIVE_CITY_NOT_WORK_CITY', 'REG_CITY_NOT_LIVE_CITY',
       'REG_CITY_NOT_WORK_CITY'],
      dtype='object')

In [19]:
class ChunkKNNImputer(BaseEstimator, TransformerMixin):
    def __init__(self, factor):
        self.imputer=KNNImputer()
        self.factor=factor
        self.features=None
    
    def fit(self, X, y=None):
        return self
        
    def fit_transform(self, X, y=None):
        self.features=X.columns
        total_idx=np.array(list(range(X.shape[0])))
        iteration=0
        rng=np.random.default_rng(seed=0)
        rng.shuffle(total_idx)
        result=None
        while(len(total_idx)>iteration*self.factor):
            curr_idx=total_idx[iteration*self.factor : (iteration+1)*self.factor]
            X_curr=X.iloc[curr_idx, :]
            if result is None:
                result=self.imputer.fit_transform(X_curr, y)
            else:
                result=np.concatenate((result, self.imputer.fit_transform(X_curr, y)), axis=0)
            iteration+=1
        print(result.shape)
        return result[total_idx.argsort(), :]

    def transform(self, X, y=None):
        return self.imputer.transform(X, y)    
            
    def get_feature_names_out(self, input_features=None):
        return self.features

            

#### This Cell will take a long time upto (3.5 minutes)

In [20]:
ct2=ColumnTransformer(transformers=[('numerical', ChunkKNNImputer(factor=10000), ['AMT_ANNUITY', 'AMT_CREDIT', 'AMT_INCOME_TOTAL', 
                                                                                     'AMT_GOODS_PRICE', 'YEARS_BIRTH', 'YEARS_ID_PUBLISH', 
                                                                                     'YEARS_REGISTRATION', 'YEARS_EMPLOYED', 'YEARS_LAST_PHONE_CHANGE', 
                                                                                     'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']),
                                    
                                    ('encodings', OneHotEncoder(sparse_output=False), ['CODE_GENDER', 'EMERGENCYSTATE_MODE', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY',
                                                                    'FONDKAPREMONT_MODE', 'HOUSETYPE_MODE', 'NAME_CONTRACT_TYPE',
                                                                    'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE',
                                                                    'NAME_INCOME_TYPE', 'NAME_TYPE_SUITE', 'OCCUPATION_TYPE',
                                                                    'ORGANIZATION_TYPE', 'WALLSMATERIAL_MODE'])],
                      remainder='passthrough', n_jobs=-1)


clean3=pd.DataFrame(ct2.fit_transform(clean2))
clean3.columns=ct2.get_feature_names_out()
clean3.rename(columns=lambda x: x[11:], inplace=True)
clean3.shape

(306295, 159)

In [21]:
clean3

,AMT_ANNUITY,AMT_CREDIT,AMT_INCOME_TOTAL,AMT_GOODS_PRICE,YEARS_BIRTH,YEARS_ID_PUBLISH,YEARS_REGISTRATION,YEARS_EMPLOYED,YEARS_LAST_PHONE_CHANGE,EXT_SOURCE_1,...,TARGET,SK_ID_CURR,FLAG_DOCUMENT_3,FLAG_DOCUMENT_6,FLAG_EMP_PHONE,FLAG_PHONE,FLAG_WORK_PHONE,LIVE_CITY_NOT_WORK_CITY,REG_CITY_NOT_LIVE_CITY,REG_CITY_NOT_WORK_CITY
0,-0.160313,-0.477814,0.429218,-0.507638,-1.505945,-0.579035,-0.380268,-0.748326,0.206498,0.083037,...,1,100002,1,0,1,1,0,0,0,0
1,0.620293,1.770503,1.220622,1.643625,0.167122,-1.79087,-1.079789,-0.511613,-0.163676,0.311267,...,0,100003,1,0,1,1,0,0,0,0
2,-1.434388,-1.166318,-1.153592,-1.10452,0.689612,-0.30672,-0.206382,-0.925324,-0.179402,0.635937,...,0,100004,0,0,1,1,1,0,0,0
3,0.193578,-0.71589,-0.362187,-0.656858,0.68022,-0.369002,1.377059,0.283588,-0.418927,0.650531,...,0,100006,1,0,1,0,0,0,0,0
4,-0.361533,-0.208082,-0.520468,-0.059976,0.892561,0.307479,-0.191892,0.283158,0.172626,0.632089,...,0,100007,0,0,1,0,0,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
306290,0.042504,-0.862876,-0.098385,-0.855819,-1.536639,-0.67047,0.985815,-0.920598,-0.835071,0.14557,...,0,456251,0,0,1,0,0,0,0,0
306291,-1.061651,-0.825231,-1.100831,-0.855819,1.08566,0.726222,-0.170014,1.067877,-1.165324,0.733055,...,0,456252,1,0,0,1,0,0,0,0
306292,0.214339,0.209343,-0.151146,0.138984,-0.24496,1.428543,0.497401,2.380926,1.144032,0.744026,...,0,456253,1,0,1,0,0,1,0,1
306293,-0.479391,-0.570318,0.059896,-0.594683,-0.933291,-1.366828,-0.68883,1.03411,-0.775794,0.423192,...,1,456254,1,0,1,0,0,0,1,1
